# Setup

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [11]:
import sys
from pathlib import Path

def find_repo_root(start: Path, marker=".git") -> Path:
    for parent in [start, *start.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find repo root from {start}")

repo_base = find_repo_root(Path.cwd())
# repo_base
tools_path = repo_base / "tools"
sys.path.append(str(tools_path))

In [12]:
from battle import *

def pull_by_id(num):
    return Battle(f'./../../data/replays/gen9-randombattle/gen9randombattle-{num}.json')

log_dir = repo_base / f"replays/gen9-randombattle/"
logs = Path('./../../data/replays/gen9-randombattle/')

In [13]:
rows = []

stat_names = ["hp","atk","def","spa","spd","spe"]

for file in logs.iterdir():
    if (file.is_file() and (file.name[0]!='.')) :
        try : 
            battle = Battle(file)
        except UnicodeDecodeError as e : 
            print(f"Could not handle {file.name} : {e}")
            continue
        if not battle.custom_ruleQ:
            to_add = { # initial non-team data we may want to train on
                "id": battle.id.removeprefix("gen9randombattle-"),
                "rated": battle.rated,
                "duration": battle.end_time - battle.start_time,
                "p1": battle.p1.name,
                "p2": battle.p2.name,
                "p1_elo" : battle.p1.elo0,
                "p2_elo": battle.p2.elo0,
                "p1_wins" : battle.p1.name == battle.winner.name,
            }

            # team data and stats for each mon
            for i in range(2):
                for j, mon_name in enumerate(battle.teams_full[i].keys()):
                    poke = battle.teams_full[i][mon_name]
                    to_add[f"p{i+1}k{j+1}_id"] = poke["speciesId"]
                    for stat in stat_names:
                        to_add[f"p{i+1}k{j+1}_{stat}"] = poke["stats"][stat]
                    try:
                        for k in range(2):
                            to_add[f"p{i+1}k{j+1}_T{k}"] = poke['types'][k] if (k < len(poke['types'])) else None
                    except KeyError as e:
                        print(f"Error with {poke["speciesId"]} in {battle.id}")
                        continue
            rows.append(to_add)

In [15]:
df = pd.DataFrame(rows)

In [16]:
df.head(15)

,id,rated,duration,p1,p2,p1_elo,p2_elo,p1_wins,p1k1_id,p1k1_hp,...,p2k5_T1,p2k6_id,p2k6_hp,p2k6_atk,p2k6_def,p2k6_spa,p2k6_spd,p2k6_spe,p2k6_T0,p2k6_T1
0,2631906096,True,598,sufideu,saberclaw,1135,1140,False,quaquaval,264,...,Fairy,klefki,233,139,201,183,194,174,Steel,Fairy
1,2631763570,False,167,PineappleCats,L4V,1959,1949,False,greninja,246,...,None,toxtricity,257,165,162,234,162,170,Electric,Poison
2,2631369343,True,275,Chicken347,cococem,1999,2068,True,munkidori,269,...,None,electrodehisui,246,92,172,189,189,311,Electric,Grass
3,2631529004,False,123,WhatEver2102,Duck Cop,1999,1982,True,sinistcha,254,...,None,terapagos,265,105,175,145,175,137,Normal,None
4,2631993792,False,301,monomythic,OverthereStair,2120,2062,False,ambipom,263,...,None,trevenant,296,247,186,166,197,150,Ghost,Grass
5,2631629401,False,116,Elite 4 Waally,jsjsiis,1958,1920,False,sudowoodo,284,...,None,probopass,257,105,316,188,325,125,Rock,Steel
6,2631479578,False,112,gamergray23,TheOldTimers,1928,1927,False,appletun,352,...,Dragon,regice,284,93,226,226,402,138,Ice,None
7,2631439736,False,448,Mr Brightside,indias last hope,2115,2259,False,clodsire,343,...,None,reuniclus,337,119,182,270,200,103,Psychic,None
8,2631771408,False,287,Illuminating_Fate,medo6037,2170,2177,True,sandyshocks,267,...,Flying,barraskewda,231,246,144,144,128,267,Water,None
9,2631528750,True,326,Emily Montes,Elite 4 Waally,1907,1937,True,staraptor,263,...,Rock,cobalion,277,149,253,190,161,219,Steel,Fighting


In [12]:
for i in range(2):
    for j in range(6):
        new_col_name = f"p{i+1}k{j+1}_atkM"
        old_col_names = [f"p{i+1}k{j+1}_atk", f"p{i+1}k{j+1}_spa"]
        matches[new_col_name] = matches[old_col_names].max(axis=1)
        matches = matches.drop(columns=old_col_names)

# EDA

In [9]:
import ydata_profiling
from ydata_profiling import ProfileReport

In [3]:
!{sys.executable} -m pip install --upgrade setuptools

In [22]:
df2 = df.drop(columns=['id','p1','p2'])

In [26]:
profile = ProfileReport(df2, title="pokemonEDA")
profile.to_file("report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████████████████| 113/113 [00:00<00:00, 198.54it/s]
/opt/homebrew/Caskroom/miniforge/base/envs/erdos_ds_environment/lib/python3.12/site-packages/ydata_profiling/model/pandas/duplicates_pandas.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index(name=duplicates_key)
/opt/homebrew/Caskroom/miniforge/base/envs/erdos_ds_environment/lib/python3.12/site-packages/ydata_profiling/model/pandas/duplicates_pandas.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_in

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]